# Hyperparameter Tuning Quickstart

This notebook demonstrates how to use the two-stage tuning module on a simulated binary classification dataset.

## Quickstart flow map

```
Your data
   │
   ├─ X_train, y_train ──────────────────────────────────────────────────┐
   └─ X_val,   y_val   ──────────────────────────────────────────────────┤
                                                                          │
   ┌──────────────────────────────────────────────────────────────────────┘
   │
   ▼
[1] randomized_search(estimator, X_train, y_train, X_val, y_val,
                       score_fn, param_distributions, n_iter=25)
      → random_results  {results, best_params, best_score, best_estimator}
   │
   ▼
[2] build_refined_grid_from_random_results(random_results,
                                            numeric_multipliers=(...),
                                            top_k_random=1)
      → refined_grid   {param_name: [candidate, ...], ...}
   │
   ▼
[3] grid_search(estimator, X_train, y_train, X_val, y_val,
                 score_fn, param_grid=refined_grid)
      → grid_results   {results, best_params, best_score, best_estimator}
   │
   ▼
 best_overall_estimator  ←  max(random best, grid best)
```

> **Shortcut:** `tune_hyperparameters(...)` runs all three steps in one call.

## Parameter reference

---

### `randomized_search(...)`

| Parameter | Type | Default | Description |
|---|---|---|---|
| `estimator` | sklearn estimator | — | Unfitted model instance. Cloned internally for each trial so the original is never mutated. |
| `X_train` / `y_train` | array-like | — | Training features and labels used to fit each trial model. |
| `X_val` / `y_val` | array-like | — | Held-out features and labels used to score each trial. Never seen during fitting. |
| `score_fn` | `callable(y_true, y_pred) → float` | — | Any function that accepts ground-truth and predicted labels and returns a scalar. Higher is better. |
| `param_distributions` | `dict[str, list \| scipy.stats rv]` | — | Parameter search space. Values can be a list of fixed choices or a `scipy.stats` distribution (e.g. `loguniform`, `randint`). |
| `n_iter` | `int` | `25` | Number of random parameter combinations to draw and evaluate. More iterations = broader coverage, slower runtime. |
| `random_state` | `int` | `42` | Seed for reproducibility of the random sampler. |

---

### `build_refined_grid_from_random_results(...)`

| Parameter | Type | Default | Description |
|---|---|---|---|
| `random_results` | `dict` | — | The dict returned by `randomized_search`. Must contain a non-empty `"results"` list. |
| `numeric_multipliers` | `tuple[float, ...]` | `(0.5, 0.75, 1.0, 1.25, 1.5)` | Multiplicative factors applied to each numeric center value from top trial(s). Controls neighborhood width. `1.0` always keeps the original value. |
| `top_k_random` | `int` | `1` | Number of top-ranked random trials to use as grid centers. `1` focuses tightly on the single best trial; higher values cover more of the parameter space. |
| `max_top_k_random` | `int` | `5` | Hard ceiling on `top_k_random`. Guards against accidentally requesting more centers than make sense. |
| `max_combinations_tested` | `int` | `300` | Soft cap on the final Cartesian product size of the refined grid. The builder trims candidate lists from the widest parameter inward until the grid fits. |

---

### `grid_search(...)`

| Parameter | Type | Default | Description |
|---|---|---|---|
| `estimator` | sklearn estimator | — | Same as in `randomized_search`. |
| `X_train` / `y_train` / `X_val` / `y_val` | array-like | — | Same as in `randomized_search`. |
| `score_fn` | callable | — | Same as in `randomized_search`. |
| `param_grid` | `dict[str, list]` | — | Explicit parameter grid. Each key maps to a list of values; all Cartesian combinations are evaluated. Typically built by `build_refined_grid_from_random_results`. |

---

### `tune_hyperparameters(...)` — wrapper (all parameters)

| Parameter | Type | Default | Description |
|---|---|---|---|
| `estimator` | sklearn estimator | — | See `randomized_search`. |
| `X_train` / `y_train` / `X_val` / `y_val` | array-like | — | See `randomized_search`. |
| `score_fn` | callable | — | See `randomized_search`. |
| `random_param_distributions` | `dict \| None` | `None` | Required when `run_randomized=True`. Passed directly to `randomized_search`. |
| `explicit_param_grid` | `dict \| None` | `None` | Optional pre-built grid for the explicit stage. If `None` and `run_explicit=True`, the grid is auto-built from randomized results. |
| `run_randomized` | `bool` | `True` | Whether to run the randomized stage. Set to `False` to skip straight to the explicit stage (requires `explicit_param_grid`). |
| `run_explicit` | `bool` | `True` | Whether to run the explicit grid stage. Set to `False` for randomized-only tuning. |
| `n_random_iterations` | `int` | `25` | Passed to `randomized_search` as `n_iter`. |
| `random_state` | `int` | `42` | Seed for the randomized stage. |
| `top_k_random` | `int` | `1` | See `build_refined_grid_from_random_results`. |
| `max_top_k_random` | `int` | `5` | See `build_refined_grid_from_random_results`. |
| `numeric_multipliers` | `tuple[float, ...]` | `(0.5, 0.75, 1.0, 1.25, 1.5)` | See `build_refined_grid_from_random_results`. |
| `max_combinations_tested` | `int` | `300` | See `build_refined_grid_from_random_results`. Also passed as the Cartesian cap to `_downsample_grid_to_max`. |
| `save_results` | `str \| None` | `None` | Filename (with or without `.pkl` extension) to persist results. `None` skips saving. |
| `overwrite` | `bool` | `False` | If `False`, appends `_1`, `_2`, ... to the filename when the target already exists, preventing accidental overwrites. |

## 1. Setup and Imports

This cell imports all libraries and tuning helpers used in the quickstart.

It imports tuning helpers from `utils.HyperparameterTuning`.

In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Import tuning functions.
from utils.HyperparameterTuning import (
        randomized_search,
        build_refined_grid_from_random_results,
        grid_search,
        tune_hyperparameters,
    )

    

## 2. Simulate Example Data

This cell creates a small synthetic binary classification dataset and wraps it in a pandas DataFrame.

The goal is to provide a quick, reproducible dataset so you can focus on tuning workflow rather than data loading.

In [2]:
# 1) Simulate a small DataFrame
X_arr, y_arr = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    class_sep=1.1,
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(X_arr.shape[1])]
df = pd.DataFrame(X_arr, columns=feature_names)
df["target"] = y_arr

df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,target
0,0.158260,-1.358451,-0.731976,1.423803,-3.844025,3.483550,-1.138616,3.785562,0.187324,-0.644494,1.415466,-1.276957,1
1,-0.992810,-4.707553,0.278002,0.828323,0.357541,1.999388,0.459291,3.186180,0.908331,2.398598,0.793877,-1.207872,1
2,-1.149757,0.757892,0.789755,-3.339486,0.400026,-0.684521,-0.404846,2.108033,5.738047,-4.362844,0.540711,-0.562600,0
3,-1.651866,-0.933370,1.427686,-1.360110,1.299783,-0.877738,-0.195328,1.939149,5.096380,-1.968965,-0.388779,1.339922,0
4,0.668624,-1.788168,1.219085,-1.060369,2.816049,-0.981300,-0.770247,3.131515,4.944811,-3.114937,0.688563,-0.803191,0


## 3. Train/Validation Split

This cell separates features and target, then creates training and validation sets.

All tuning functions in this module use train data for fitting and validation data for scoring.

In [3]:
# 2) Train/validation split
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_val.shape

((900, 12), (300, 12))

## 4. Define Model, Metric, and Random Search Space

This cell sets the baseline estimator (`LogisticRegression`), the score function (`F1`), and randomized parameter distributions.

Use this section to customize what the tuner explores.

In [4]:
# 3) Define estimator, score function, and random distributions
base_model = LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)

def score_fn(y_true, y_pred):
    return f1_score(y_true, y_pred)

random_param_distributions = {
    "C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
    "penalty": ["l1", "l2"],
    "class_weight": [None, "balanced"]
}

## 5. Run Randomized Search (Ballpark Stage)

This cell performs quick stochastic exploration and returns:

- trial-level scores and parameters
- best randomized score
- best randomized parameter set

In [5]:
# 4) Randomized stage
random_results = randomized_search(
    estimator=base_model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    score_fn=score_fn,
    param_distributions=random_param_distributions,
    n_iter=12,
    random_state=42,
)

{
    "best_score": random_results["best_score"],
    "best_params": random_results["best_params"],
    "n_trials": random_results["n_trials"],
}

{'best_score': 0.8553054662379421,
 'best_params': {'penalty': 'l2', 'class_weight': None, 'C': 10.0},
 'n_trials': 12}

## 6. Build Refined Grid and Run Explicit Search

This cell converts top randomized results into a deterministic grid and evaluates all explicit combinations.

Use this stage when you want tighter control and clearer parameter-by-parameter comparison.

In [6]:
# 5) Build explicit grid from top randomized result(s), then run explicit grid stage
refined_grid = build_refined_grid_from_random_results(
    random_results=random_results,
    numeric_multipliers=(0.5, 0.75, 1.0, 1.25, 1.5),
    top_k_random=1,
    max_top_k_random=5,
    max_combinations_tested=120,
)

grid_results = grid_search(
    estimator=base_model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    score_fn=score_fn,
    param_grid=refined_grid,
)

{
    "refined_grid": refined_grid,
    "best_score": grid_results["best_score"],
    "best_params": grid_results["best_params"],
    "n_trials": grid_results["n_trials"],
}

{'refined_grid': {'penalty': ['l2'],
  'class_weight': [None],
  'C': [5.0, 7.5, 10.0, 12.5, 15.0]},
 'best_score': 0.8553054662379421,
 'best_params': {'C': 5.0, 'class_weight': None, 'penalty': 'l2'},
 'n_trials': 5}

## 7. Use the One-Call Wrapper

This cell runs the full two-stage flow (`randomized -> refined grid -> explicit`) and optionally saves results.

The returned dictionary includes stage outputs plus the overall best model configuration.

In [7]:
# 6) One-call wrapper: randomized + explicit, then save results
full_results = tune_hyperparameters(
    estimator=base_model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    score_fn=score_fn,
    random_param_distributions=random_param_distributions,
    explicit_param_grid=None,
    run_randomized=True,
    run_explicit=True,
    n_random_iterations=10,
    random_state=42,
    top_k_random=1,
    max_top_k_random=5,
    max_combinations_tested=100,
    save_results="hyperparameter_tuning_demo.pkl",
    overwrite=False,
)

{
    "best_overall_score": full_results["best_overall_score"],
    "best_overall_params": full_results["best_overall_params"],
    "save_path": full_results["save_path"],
    "total_elapsed_seconds": full_results["total_elapsed_seconds"],
}

{'best_overall_score': 0.8553054662379421,
 'best_overall_params': {'penalty': 'l2', 'class_weight': None, 'C': 10.0},
 'save_path': 'hyperparameter_tuning_demo.pkl',
 'total_elapsed_seconds': 0.09105519999866374}

## 8. Interpret Trial-Level Results

This final cell turns the explicit-stage results into a DataFrame for inspection.

Suggested checks:

- Sort by score to see strongest parameter combinations.
- Compare elapsed time by trial to identify expensive settings.
- Use top rows as candidates for deeper validation.

In [8]:
# 7) Inspect per-trial output records from the explicit stage
pd.DataFrame(full_results["grid"]["results"]).head(10)

,trial_index,params,score,elapsed_seconds,model_name
0,1,"{'C': 5.0, 'class_weight': None, 'penalty': 'l2'}",0.855305,0.003315,LogisticRegression
1,2,"{'C': 7.5, 'class_weight': None, 'penalty': 'l2'}",0.855305,0.003849,LogisticRegression
2,3,"{'C': 10.0, 'class_weight': None, 'penalty': '...",0.855305,0.003985,LogisticRegression
3,4,"{'C': 12.5, 'class_weight': None, 'penalty': '...",0.855305,0.003487,LogisticRegression
4,5,"{'C': 15.0, 'class_weight': None, 'penalty': '...",0.855305,0.003359,LogisticRegression


---

## Common mistakes / troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: param_distributions must be a non-empty dict` | Passed `{}` or `None` to `randomized_search` | Provide at least one distribution, e.g. `{"C": loguniform(1e-3, 1e2)}` |
| `ValueError: top_k_random cannot exceed 5` | `top_k_random` set too high | Lower it, or raise `max_top_k_random` |
| `ValueError: Requested top_k_random=N but only M trials available` | `n_iter` is smaller than `top_k_random` | Increase `n_iter` or decrease `top_k_random` |
| `score_fn` returns `NaN` or crashes | Score function receives wrong arguments | Confirm signature `score_fn(y_true, y_pred) → float` |
| `RuntimeError: Could not reduce parameter grid` | `max_combinations_tested` too small for the parameter space | Raise `max_combinations_tested` or reduce `numeric_multipliers` |
| Scipy type-stub warnings in IDE | Type stubs for `scipy.stats` | Not a runtime error — safe to ignore |
| Saved `.pkl` grows unexpectedly large | `best_estimator` stored inside results | Expected — the fitted model is serialized. Filter it out before saving if size matters |

**Quick sanity-check pattern**

```python
# Smoke-test with n_iter=3 before a real run
test = randomized_search(
    base_model, X_train, y_train, X_val, y_val,
    score_fn=f1_score, param_distributions=random_param_distributions,
    n_iter=3,
)
print(test["best_score"], test["best_params"])
```

---

## Next steps — applying to a real dataset

1. **Swap in your data**  
   Replace the `make_classification` block with your own `X_train`, `y_train`, `X_val`, `y_val`.  
   The rest of the notebook works unchanged.

2. **Swap the estimator**  
   Replace `LogisticRegression()` with any sklearn-compatible model:
   ```python
   from sklearn.ensemble import RandomForestClassifier
   base_model = RandomForestClassifier(random_state=42)
   ```

3. **Adjust distributions for the new model**  
   Pick distributions that match the model's param types — e.g. `randint(10, 300)` for `n_estimators`, `uniform(0, 1)` for `max_features`.

4. **Tune the neighborhood width**  
   `numeric_multipliers` controls how wide the refined grid is around each top-trial center.  
   A tighter range (e.g. `(0.8, 0.9, 1.0, 1.1, 1.2)`) is useful when randomized results are already well-converged.

5. **Persist results for later use**  
   ```python
   full_results = tune_hyperparameters(..., save_results="my_model_tune.pkl")
   # Load later:
   import pickle
   with open("my_model_tune.pkl", "rb") as f:
       saved = pickle.load(f)
   best_params = saved["results"]["best_overall_params"]
   ```

6. **Increase randomized coverage for complex spaces**  
   Raise `n_random_iterations` (e.g. 100–200) before narrowing with the grid stage.  
   Use `top_k_random=3` to build a grid that covers multiple promising regions.